# Proyecto Final — Módulo 6

## Tema: Población adulta estadounidense (1994) con ingreso anual > $50K USD

| **Campo** | **Detalle** |
|---|---|
| **Grupo** | 2 |
| **Integrantes** | Cinthia Montero, Sebastián Calvo |
| **Curso** | Ciencia de Datos |
| **Fecha de Entrega** | 04 Setiembre 2026 |

---


# Problema:  
# Averigüar las características de la poblacióna adulta estadounidense del año 1994 que tenía ingresos anuales mayores a $ 50K USD y predecir sus futuros comportamientos.

# Objetivo:
# Identificar características que sean claves para determinar patrones que desarrollen individuos de la población que generen un ingreso anual mayor a $ 50K USD.

# **ETAPAS :**

# 2. Ingesta

In [2]:
# Librerías para manipulación y análisis de datos

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Permite visualizar todas las columnas del DataFrame
pd.set_option('display.max_columns', None)

import os
os.environ["OMP_NUM_THREADS"] = "5"

In [4]:
# Instalamos 

!pip install ucimlrepo


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# Descargamos dataset de https://archive.ics.uci.edu/dataset/2/adult?utm_source=chatgpt.com

from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
adult = fetch_ucirepo(id=2) 
  
# data (as pandas dataframes) 
X = adult.data.features 
y = adult.data.targets 
  
# metadata 
print(adult.metadata) 
  
# variable information 
print(adult.variables) 


{'uci_id': 2, 'name': 'Adult', 'repository_url': 'https://archive.ics.uci.edu/dataset/2/adult', 'data_url': 'https://archive.ics.uci.edu/static/public/2/data.csv', 'abstract': 'Predict whether annual income of an individual exceeds $50K/yr based on census data. Also known as "Census Income" dataset. ', 'area': 'Social Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 48842, 'num_features': 14, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Income', 'Education Level', 'Other', 'Race', 'Sex'], 'target_col': ['income'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1996, 'last_updated': 'Tue Sep 24 2024', 'dataset_doi': '10.24432/C5XW20', 'creators': ['Barry Becker', 'Ronny Kohavi'], 'intro_paper': None, 'additional_info': {'summary': "Extraction was done by Barry Becker from the 1994 Census database.  A set of reasonably clean records was extracted using the fol

# 3. Data Quality

## 3.1 Identificación de valores y tipos de dato.

Primeramente identificamos valores faltantes, simbolos insuales, tipos de dato incongrüentes y valores nulos.

Para este caso en particular al tratarse de un Censo se asume que los valores ya fueron previamente revisados y no se estan duplicando datos. Este dataset no cuenta con alguna columna con datos de identificación ó datos singulares y únicos que permitan confirmar que la información no es duplicada.  No obstante se hace una revisión para hacer constar el peso de los datos duplicados sobre el total del dataset.


In [ ]:
# Redondeamos a dos decimales
pd.set_option('display.float_format', '{:.2f}'.format)

In [63]:
# Visualizamos las 5 filas para validación

df = pd.concat([X, y], axis=1)
display(df.sample(5))

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
27238,46,Private,169953,Some-college,10,Divorced,Exec-managerial,Not-in-family,White,Female,0,0,40,United-States,<=50K
2454,48,Private,207058,HS-grad,9,Divorced,Adm-clerical,Unmarried,White,Female,0,0,37,United-States,<=50K
16726,35,Private,214896,HS-grad,9,Divorced,Adm-clerical,Not-in-family,White,Female,0,0,45,United-States,<=50K
8186,21,Private,118657,HS-grad,9,Separated,Machine-op-inspct,Other-relative,White,Male,0,0,40,United-States,<=50K
33444,19,Local-gov,259169,Some-college,10,Never-married,Prof-specialty,Own-child,White,Female,0,0,30,United-States,<=50K.


In [31]:
# Dimensiones del dataset

print(f"El dataset contiene {df.shape[0]} filas y {df.shape[1]} columnas.")    # cantidad de filas y columnas

El dataset contiene 48842 filas y 15 columnas.


In [32]:
# Nombres de las columnas del dataset
df.columns.tolist()

['age',
 'workclass',
 'fnlwgt',
 'education',
 'education-num',
 'marital-status',
 'occupation',
 'relationship',
 'race',
 'sex',
 'capital-gain',
 'capital-loss',
 'hours-per-week',
 'native-country',
 'income']

In [33]:
# Información general y tipos de datos
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             48842 non-null  int64 
 1   workclass       47879 non-null  object
 2   fnlwgt          48842 non-null  int64 
 3   education       48842 non-null  object
 4   education-num   48842 non-null  int64 
 5   marital-status  48842 non-null  object
 6   occupation      47876 non-null  object
 7   relationship    48842 non-null  object
 8   race            48842 non-null  object
 9   sex             48842 non-null  object
 10  capital-gain    48842 non-null  int64 
 11  capital-loss    48842 non-null  int64 
 12  hours-per-week  48842 non-null  int64 
 13  native-country  48568 non-null  object
 14  income          48842 non-null  object
dtypes: int64(6), object(9)
memory usage: 5.6+ MB


## 3.2 Valores Duplicados 

In [77]:
# Duplicados exactos (todas las columnas, incluyendo fnlwgt)
duplicados = df[df.duplicated(keep=False)].sort_values(by=df.columns.tolist())

total_duplicados = df.duplicated().sum()  # solo las copias "de más"
total_filas_involucradas = duplicados.shape[0]  # original + copia

print(f'Total de filas duplicadas (copias de más): {total_duplicados}')
print(f'Peso relativo sobre el total del dataset: {total_duplicados / len(df) * 100:.2f}%')
print()

duplicados.head(8)

Total de filas duplicadas (copias de más): 29
Peso relativo sobre el total del dataset: 0.06%



,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
36461,18,Self-emp-inc,378036,12th,8,Never-married,Farming-fishing,Own-child,White,Male,0,0,10,United-States,<=50K.
48521,18,Self-emp-inc,378036,12th,8,Never-married,Farming-fishing,Own-child,White,Male,0,0,10,United-States,<=50K.
17673,19,Private,97261,HS-grad,9,Never-married,Farming-fishing,Not-in-family,White,Male,0,0,40,United-States,<=50K
18698,19,Private,97261,HS-grad,9,Never-married,Farming-fishing,Not-in-family,White,Male,0,0,40,United-States,<=50K
6990,19,Private,138153,Some-college,10,Never-married,Adm-clerical,Own-child,White,Female,0,0,10,United-States,<=50K
21318,19,Private,138153,Some-college,10,Never-married,Adm-clerical,Own-child,White,Female,0,0,10,United-States,<=50K
15189,19,Private,146679,Some-college,10,Never-married,Exec-managerial,Own-child,Black,Male,0,0,30,United-States,<=50K
21490,19,Private,146679,Some-college,10,Never-married,Exec-managerial,Own-child,Black,Male,0,0,30,United-States,<=50K


### Hallazgos sobre valores duplicados:

Se encuentran 29 filas que parecen ser duplicadas, sin embargo no se cuenta con el suficiente fundamento para determinar si realmente son valores duplicados ó bien se trata de registros identicos mas no duplicados. Se deciden conservar aquellas filas que parecen duplicadas.

## 3.3 Valores Nulos

Se valida la existencia de valores nulos, revisamos tanto el formato del valor nulo convertido mediante repositorio **ucimlrepo** como también su formato original.

Detectamos que la librería de origen no normalizó el símbolo de faltante de forma consistente, por lo que unificamos ambas codificaciones antes de continuar.

Revisaremos la existencia de los siguientes símbolos que representan en este dataset valores nulos ó faltantes: "NaN", "?" y " ".

In [34]:
# Cantidad de nulos por columna
df.isnull().sum()

# Solo las columnas que sí tienen nulos, con porcentaje
nulos = df.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
pd.DataFrame({
    'Valores faltantes': nulos,
    'Porcentaje (%)': (nulos / len(df) * 100).round(2)
})

,Valores faltantes,Porcentaje (%)
occupation,966,1.98
workclass,963,1.97
native-country,274,0.56


In [35]:
# Verificación de nulos en formato original de dataset.

(df == '?').sum()[lambda x: x > 0]

workclass         1836
occupation        1843
native-country     583
dtype: int64

In [36]:
# Busca strings vacíos o que son solo espacios en las columnas de texto
cols_obj = df.select_dtypes(include='object').columns

vacios = {}
for col in cols_obj:
    n = (df[col].astype(str).str.strip() == '').sum()
    if n > 0:
        vacios[col] = n

print('Columnas con celdas vacías/solo espacios:', vacios or 'ninguna')

Columnas con celdas vacías/solo espacios: ninguna


In [37]:
nulos_nan = df.isnull().sum()
nulos_signo = (df == '?').sum()
total_faltantes = nulos_nan.add(nulos_signo, fill_value=0)
total_faltantes = total_faltantes[total_faltantes > 0].sort_values(ascending=False)
pd.DataFrame({
    'Valores faltantes': total_faltantes.astype(int),
    'Porcentaje (%)': (total_faltantes / len(df) * 100).round(2)
})

,Valores faltantes,Porcentaje (%)
occupation,2809,5.75
workclass,2799,5.73
native-country,857,1.75


In [72]:
pd.set_option('display.max_columns', None)

df[df[['workclass','occupation','native-country']].isnull().any(axis=1) |
   (df[['workclass','occupation','native-country']] == '?').any(axis=1)].sample(5)

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
27331,61,?,42938,Bachelors,13,Never-married,?,Not-in-family,White,Male,0,0,7,United-States,>50K
33248,20,Private,346341,Some-college,10,Never-married,Adm-clerical,Not-in-family,White,Female,0,0,40,NaN,<=50K.
18750,19,?,234519,Some-college,10,Never-married,?,Own-child,White,Male,0,0,35,United-States,<=50K
25522,66,?,108185,9th,5,Married-civ-spouse,?,Husband,Black,Male,0,0,40,United-States,<=50K
44582,56,NaN,141076,HS-grad,9,Divorced,NaN,Not-in-family,Black,Female,3674,0,40,United-States,<=50K.


### Hallazgos de valores faltantes: 

Se encuentran 6,465 celdas con valores nulos. Sin embargo por categoría los valores nulos representan aproximadamente entre un 5% y un 6% para las columnas de "occupation" y "worldclass", y para la columna "native-country" este representa apenas un 1.75%.   

## 3.4 Imputación

Debido a los valores faltantes representan aproximadamente un 5% aproximadamente ó inclusive un valor menor del total de cada columna y a la vez al tratarse de datos de tipo cualitativos, **imputaremos con base en la moda**.

In [39]:
cols_con_nulos = ['workclass', 'occupation', 'native-country']

# Moda de cada columna (ignora nulos y "?")
modas = {}
for col in cols_con_nulos:
    moda = df.loc[~df[col].isnull() & (df[col] != '?'), col].mode()[0]
    modas[col] = moda
    print(f'{col}: moda = {moda}')

workclass: moda = Private
occupation: moda = Prof-specialty
native-country: moda = United-States


In [40]:
# Imputación: reemplaza NaN y "?" por la moda de cada columna
df_clean = df.copy()

for col in cols_con_nulos:
    df_clean[col] = df_clean[col].replace('?', modas[col])
    df_clean[col] = df_clean[col].fillna(modas[col])

# Verificación
print(df_clean[cols_con_nulos].isnull().sum())
print((df_clean[cols_con_nulos] == '?').sum())

workclass         0
occupation        0
native-country    0
dtype: int64
workclass         0
occupation        0
native-country    0
dtype: int64


Imputacion de punto (.) al final del valor en columna "income".

In [41]:
# Quita el punto final para unificar "<=50K." con "<=50K" y ">50K." con ">50K"
df['income'] = df['income'].str.rstrip('.')

# Verificación
df['income'].unique()

array(['<=50K', '>50K'], dtype=object)

In [42]:
df_clean['income'] = df_clean['income'].str.rstrip('.')
df_clean['income'].unique()

array(['<=50K', '>50K'], dtype=object)

In [73]:
df_clean.sample(5)

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
9874,55,Self-emp-inc,209569,HS-grad,9,Divorced,Sales,Unmarried,White,Female,0,0,50,United-States,>50K
22228,56,Private,187295,Bachelors,13,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,40,United-States,>50K
14523,44,Private,125461,Bachelors,13,Married-civ-spouse,Sales,Husband,White,Male,0,0,40,United-States,>50K
44602,34,Private,176862,Bachelors,13,Never-married,Prof-specialty,Not-in-family,White,Male,0,0,40,United-States,<=50K
3037,42,Private,307638,HS-grad,9,Divorced,Adm-clerical,Own-child,White,Female,0,0,40,United-States,<=50K


### Hallazgos de Imputación:

Se imputan los datos nulos identificados tanto con simbolo "NaN" como con simbolo "?", dejando en cero la cantidad de datos nulos.

También se limpia un punto (.) en la columna "income" reduciendo la cantidad de categorías para esa variable a dos únicamente.

## 3.5 Identificación de tipos de variables :

### Variables cuantitativas:

| N | Column | Non-Null Count | Dtype |
|---|---|---|---|
| 0 | age | 48842 non-null | int64 |
| 2 | fnlwgt | 48842 non-null | int64 |
| 4 | education-num | 48842 non-null | int64 |
| 10 | capital-gain | 48842 non-null | int64 |
| 11 | capital-loss | 48842 non-null | int64 |
| 12 | hours-per-week | 48842 non-null | int64 |

### Variables cualitativas:

| N | Column | Non-Null Count | Dtype |
|---|---|---|---|
| 1 | workclass | 47879 non-null | object |
| 3 | education | 48842 non-null | object |
| 5 | marital-status | 48842 non-null | object |
| 6 | occupation | 47876 non-null | object |
| 7 | relationship | 48842 non-null | object |
| 8 | race | 48842 non-null | object |
| 9 | sex | 48842 non-null | object |
| 13 | native-country | 48568 non-null | object |
| 14 | income | 48842 non-null | object |

### 3.5.1 Estadística descriptiva para variables cuantitativas: 





In [64]:
# Lista de variables cuantitativas
vars_cuantitativas = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']



In [ ]:
# Estadística descriptiva de variables cuantitativas:

resumen = df[vars_cuantitativas].describe().T
resumen['rango'] = resumen['max'] - resumen['min']
resumen['mediana'] = df[vars_cuantitativas].median()
resumen

,count,mean,std,min,25%,50%,75%,max,rango,mediana
age,48842.00,38.64,13.71,17.00,28.00,37.00,48.00,90.00,73.00,37.00
fnlwgt,48842.00,189664.13,105604.03,12285.00,117550.50,178144.50,237642.00,1490400.00,1478115.00,178144.50
education-num,48842.00,10.08,2.57,1.00,9.00,10.00,12.00,16.00,15.00,10.00
capital-gain,48842.00,1079.07,7452.02,0.00,0.00,0.00,0.00,99999.00,99999.00,0.00
capital-loss,48842.00,87.50,403.00,0.00,0.00,0.00,0.00,4356.00,4356.00,0.00
hours-per-week,48842.00,40.42,12.39,1.00,40.00,40.00,45.00,99.00,98.00,40.00


### Descripción de las variables cuantitativas:

Se reconocen 6 variables cuantitativas, no se identifica la necesidad de convertir a variable cualitativa alguna de las existentes.

- **age**: su rango va de 17 a 90 años, con un promedio de 38.6 y mediana 37, bastante cercanas, lo que indica una distribución razonablemente simétrica.

- **fnlwgt**: rango muy amplio (12.285 a 1.490.400) y media (189.664) bastante más alta que la mediana (178.144). Distribución asimétrica cesgada a la derecha.

- **education-num**: va de 1 a 16, la media es 10.08 y la mediana 10, se comporta de manera simétrica. 

- **capital-gain**: el 91.7% de las personas tiene capital-gain igual a 0. La media (1079) está muy por encima de la mediana (0). Distribución asimétrica cesgada a la derecha.

- **capital-loss**: 95.3% en cero, media 87.5 contra una mediana de 0. Distribución asimétrica cesgada a la derecha.

- **hours-per-week**: es una variable con distribución bastante simétrico.

### 3.5.2 Estadística descriptiva para variables cualitativas:

In [79]:
vars_cualitativas = ['workclass', 'education', 'marital-status', 'occupation',
                      'relationship', 'race', 'sex', 'native-country', 'income']

resumen_cat = df_clean[vars_cualitativas].describe().T
resumen_cat['freq'] = pd.to_numeric(resumen_cat['freq'])
resumen_cat['count'] = pd.to_numeric(resumen_cat['count'])
resumen_cat['unique'] = pd.to_numeric(resumen_cat['unique'])

resumen_cat['% de la moda'] = (resumen_cat['freq'] / len(df_clean) * 100).round(2)
resumen_cat

,count,unique,top,freq,% de la moda
workclass,48842,8,Private,36705,75.15
education,48842,16,HS-grad,15784,32.32
marital-status,48842,7,Married-civ-spouse,22379,45.82
occupation,48842,14,Prof-specialty,8981,18.39
relationship,48842,6,Husband,19716,40.37
race,48842,5,White,41762,85.50
sex,48842,2,Male,32650,66.85
native-country,48842,41,United-States,44689,91.50
income,48842,2,<=50K,37155,76.07


### Descripción de las estadísticas cualitativas por variable:

Se identificaron 9 variables cualitativas a las cuales se les analizó las siguientes estadísticas: dato moda para cada variable cualitativa, la frecuencia absoluta, y cuánto representa el dato moda del todal de datos por variable:



- **workclass**: 8 categorías, dominada 75.15% por "Private", es la categoría claramente mayoritaria del mundo laboral en EE.UU. de 1994. 

- **education**: 16 categorías, la moda es "HS-grad" con solo un 32.32% del total de registros.

- **marital-status**: 7 categorías, la moda fue "Married-civ-spouse" con 45.82% del total de registros. Casi la mitad de la población se encuentra formando parte de un matrimonio.

- **occupation**: 14 categorías, "Prof-specialty" es la moda con apenas un 18% del total de registros, es una variable categórica que se ha repartido de manera más equitativa.

- **relationship**: 6 categorías, la moda es: "Husband" 40%, casi la mitad de la población aproximadamente. 

- **race**: 5 categorías, "White" con 85.5% de la población encuestada en el año 1994 en Estados Unidos era de etnia blanca. 

- **sex**: binaria, La moda fue "Male" 66.85% del total de personas.

- **native-country**: 41 categorías, dominada 91.5% por "United-States". El resto de los 41 países se reparten menos del 9%.

- **income**: 76.07% de la población tiene un ingreso anual menor ó igual a los 50 mil dólares. 